# AI Security Vulnerability Scanning Lab

This lab demonstrates how to analyze AI/ML code for security vulnerabilities using four tools:

- **Bandit** — Python static analysis
- **Semgrep** — pattern-based SAST
- **Pylint** — code quality and correctness
- **Safety** — dependency vulnerability scanning

You will:
1. Run each tool
2. Parse the results
3. Extract severity information
4. Compare tool behavior
5. Summarize findings

This notebook assumes your project files (e.g., `demo_code_1.py`, `demo_code_2.py`) are in the same directory as this notebook.

In [ ]:
# Install required tools
!pip install bandit semgrep safety pylint -q

In [ ]:
# Show project files in the current directory
import os
os.listdir('.')

## Bandit — Static Analysis of Python Code

In [ ]:
# Run Bandit on the current directory and save JSON output
!bandit -r . -f json -o bandit.json

In [ ]:
# Load Bandit results
import json

with open("bandit.json") as f:
    bandit = json.load(f)

len(bandit.get("results", []))

In [ ]:
# Display Bandit findings (severity, rule, CWE, file, line, text)
bandit_findings = [
    {
        "severity": r.get("issue_severity"),
        "rule": r.get("test_id"),
        "cwe": r.get("issue_cwe", {}).get("id"),
        "file": r.get("filename"),
        "line": r.get("line_number"),
        "text": r.get("issue_text")
    }
    for r in bandit.get("results", [])
]
bandit_findings[:10]  # show first 10

In [ ]:
# Bandit severity counts
import pandas as pd

sev_series = pd.Series([r.get("issue_severity") for r in bandit.get("results", [])])
sev_series.value_counts()

## Semgrep — Pattern-Based SAST

In [ ]:
# Run Semgrep with auto config and save JSON output
!semgrep --config auto . --json --output semgrep.json

In [ ]:
# Load Semgrep results
with open("semgrep.json") as f:
    semgrep = json.load(f)

len(semgrep.get("results", []))

In [ ]:
# Display Semgrep findings (rule, file, line, message)
semgrep_findings = [
    {
        "rule": r.get("check_id"),
        "file": r.get("path"),
        "line": r.get("start", {}).get("line"),
        "message": r.get("extra", {}).get("message")
    }
    for r in semgrep.get("results", [])
]
semgrep_findings[:10]

## Pylint — Code Quality and Correctness

In [ ]:
# Run Pylint with warnings and errors enabled, save to text file
!pylint *.py --disable=all --enable=W,E > pylint.txt || true

In [ ]:
# Show Pylint warnings and errors
with open("pylint.txt") as f:
    pylint_output = f.read()

print(pylint_output)

## Safety — Dependency Vulnerability Scanning

In [ ]:
# Run Safety and save JSON output
!safety check --json > safety.json || true

In [ ]:
# Load Safety results
with open("safety.json") as f:
    safety = json.load(f)

safety.get("vulnerabilities", [])

## Consolidated Analysis

In this section, we interpret the results from all tools:

- **Bandit**: Python static analysis, reporting severity levels (LOW, MEDIUM, HIGH) and CWE IDs.
- **Semgrep**: Pattern-based rules that reliably flag risky constructs such as unsafe `pickle` usage.
- **Pylint**: Code quality and correctness issues that can indirectly affect security.
- **Safety**: Dependency vulnerabilities (CVEs) in installed packages.

You can now use the `bandit_findings`, `semgrep_findings`, `pylint_output`, and `safety` objects to build tables, summaries, or reports.

In [ ]:
# Quick consolidated counts
print("Bandit findings:", len(bandit_findings))
print("Semgrep findings:", len(semgrep_findings))
print("Safety vulnerabilities:", len(safety.get("vulnerabilities", [])))

## Summary

This notebook demonstrated how to:

- Run Bandit, Semgrep, Pylint, and Safety on an AI/ML codebase.
- Parse and inspect their outputs programmatically.
- Compare how different tools detect and classify security issues.

You can now extend this notebook by:
- Mapping findings to CWE and OWASP categories.
- Generating markdown or HTML reports.
- Integrating similar checks into CI/CD pipelines.